# Repository-Level Reporting Behavior Visual

This notebook analyzes reporting behavior at the repository level.

Goal:

- Identify repositories with only silent fixes.
- Identify repositories with only transparent fixes.
- Identify repositories with a mix of silent and transparent fixes.
- Create visuals suitable for the RQ1 prevalence section.

Input:

```text
../../data/rq1/corrected_resolved_links_v2.csv
```

Additional output:

- Add severity distribution for each repository reporting behavior group.


In [10]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


In [11]:
DATA_PATH = Path('../../data/rq1/corrected_resolved_links_v2.csv')
SEVERITY_PATH = Path('../../data/rq1/repo_with_severity.csv')

df = pd.read_csv(DATA_PATH)
severity = pd.read_csv(SEVERITY_PATH)

print('reporting data:', df.shape)
print('Unique CVEs:', df['CVE_ID'].nunique())
print('severity data:', severity.shape)
print('Unique severity CVEs:', severity['CVE_ID'].nunique())
df.head()


reporting data: (832, 5)
Unique CVEs: 832
severity data: (832, 10)
Unique severity CVEs: 832


,CVE_ID,Commit Message,Links in Message,Link Presence,PATCH
0,CVE-2013-4600,Fixed some XSS problems (github issue #173),https://github.com/alkacon/opencms-core/issues...,contains links,https://github.com/alkacon/opencms-core/commit...
1,CVE-2018-3831,Apply settings filter to get cluster settings ...,https://github.com/elastic/elasticsearch/pull/...,contains links,https://github.com/elastic/elasticsearch/commi...
2,CVE-2013-4310,Changes archetypes version to match latest rel...,https://svn.apache.org/repos/asf/struts/struts...,contains links,https://github.com/apache/struts/commit/0c8366...
3,CVE-2018-1000615,"Fix for OS-12, NumberFormatException on badly ...",NaN,no links,https://github.com/opennetworkinglab/onos/comm...
4,CVE-2022-23596,fix: invalid subheader type would throw npe an...,https://github.com/junrar/junrar/issues/73,contains links,https://github.com/junrar/junrar/commit/7b16b3...


In [12]:
def extract_repo(patch):
    if pd.isna(patch):
        return None
    match = re.search(r'https://github\.com/([^/]+/[^/\s]+)', str(patch))
    return match.group(1).lower() if match else None

df['repository'] = df['PATCH'].apply(extract_repo)
df['reporting_visibility'] = df['Link Presence'].map({
    'contains links': 'Transparent',
    'no links': 'Silent',
})

# Add severity per CVE.
df = df.merge(severity[['CVE_ID', 'Severity']], on='CVE_ID', how='left')

print('Missing repository:', df['repository'].isna().sum())
print('Missing reporting_visibility:', df['reporting_visibility'].isna().sum())
print('Missing Severity:', df['Severity'].isna().sum())
print('Unique repositories:', df['repository'].nunique())
df[['CVE_ID', 'repository', 'reporting_visibility', 'Severity']].head()


Missing repository: 0
Missing reporting_visibility: 0
Missing Severity: 0
Unique repositories: 257


,CVE_ID,repository,reporting_visibility,Severity
0,CVE-2013-4600,alkacon/opencms-core,Transparent,Medium
1,CVE-2018-3831,elastic/elasticsearch,Transparent,Medium
2,CVE-2013-4310,apache/struts,Transparent,Medium
3,CVE-2018-1000615,opennetworkinglab/onos,Silent,Medium
4,CVE-2022-23596,junrar/junrar,Transparent,Medium


In [13]:
repo_counts = (
    df
    .groupby(['repository', 'reporting_visibility'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['Silent', 'Transparent']:
    if col not in repo_counts.columns:
        repo_counts[col] = 0

repo_counts['total_cves'] = repo_counts['Silent'] + repo_counts['Transparent']

repo_counts['repo_reporting_behavior'] = 'Mixed silent and transparent fixes'
repo_counts.loc[(repo_counts['Silent'] > 0) & (repo_counts['Transparent'] == 0), 'repo_reporting_behavior'] = 'Only silent fixes'
repo_counts.loc[(repo_counts['Silent'] == 0) & (repo_counts['Transparent'] > 0), 'repo_reporting_behavior'] = 'Only transparent fixes'

repo_counts = repo_counts.sort_values(['total_cves', 'repository'], ascending=[False, True]).reset_index(drop=True)
repo_counts.head(20)


reporting_visibility,repository,Silent,Transparent,total_cves,repo_reporting_behavior
0,apache/tomcat,15,52,67,Mixed silent and transparent fixes
1,apache/struts,19,24,43,Mixed silent and transparent fixes
2,spring-projects/spring-framework,20,4,24,Mixed silent and transparent fixes
3,keycloak/keycloak,18,4,22,Mixed silent and transparent fixes
4,x-stream/xstream,15,7,22,Mixed silent and transparent fixes
5,eclipse/jetty.project,5,14,19,Mixed silent and transparent fixes
6,apache/cxf,6,11,17,Mixed silent and transparent fixes
7,bcgit/bc-java,16,0,16,Only silent fixes
8,undertow-io/undertow,12,4,16,Mixed silent and transparent fixes
9,apache/activemq,2,13,15,Mixed silent and transparent fixes


In [14]:
repo_behavior_summary = (
    repo_counts['repo_reporting_behavior']
    .value_counts()
    .rename_axis('Repository reporting behavior')
    .reset_index(name='Repository count')
)
repo_behavior_summary['%'] = (repo_behavior_summary['Repository count'] / repo_counts['repository'].nunique() * 100).round(2)

# Severity distribution is counted at the CVE level within each repository behavior group.
repo_behavior_for_merge = repo_counts[['repository', 'repo_reporting_behavior']]
df_with_repo_behavior = df.merge(repo_behavior_for_merge, on='repository', how='left')

severity_order = ['Critical', 'High', 'Medium', 'Low', 'Unknown']

def format_severity_distribution(values):
    counts = values.value_counts(dropna=False).to_dict()
    parts = []

    for severity_label in severity_order:
        count = int(counts.get(severity_label, 0))
        if severity_label == 'Unknown':
            if count > 0:
                parts.append(f'{severity_label} ({count})')
        else:
            parts.append(f'{severity_label} ({count})')

    for severity_label, count in sorted(counts.items()):
        if severity_label not in severity_order:
            parts.append(f'{severity_label} ({int(count)})')

    return ', '.join(parts)

severity_distribution = (
    df_with_repo_behavior
    .groupby('repo_reporting_behavior')['Severity']
    .apply(format_severity_distribution)
    .rename('Severity distribution')
    .reset_index()
    .rename(columns={'repo_reporting_behavior': 'Repository reporting behavior'})
)

repo_behavior_summary = repo_behavior_summary.merge(
    severity_distribution,
    on='Repository reporting behavior',
    how='left',
)

repo_behavior_summary


,Repository reporting behavior,Repository count,%,Severity distribution
0,Only transparent fixes,110,42.80,"Critical (7), High (46), Medium (130), Low (12)"
1,Only silent fixes,96,37.35,"Critical (2), High (24), Medium (116), Low (5)"
2,Mixed silent and transparent fixes,51,19.84,"Critical (27), High (70), Medium (362), Low (3..."


## Publication-ready repository behavior table

The previous summary is useful for inspection, but it mixes repository counts with CVE-level severity distribution. The table below separates repository-level and CVE-level quantities:

- `Repositories` and `Repo %` are repository-level.
- `CVEs`, `CVE %`, and severity columns are CVE-level within each repository behavior group.


In [15]:
behavior_order = [
    'Only transparent fixes',
    'Only silent fixes',
    'Mixed silent and transparent fixes',
]

repo_behavior_for_merge = repo_counts[['repository', 'repo_reporting_behavior']]
df_with_repo_behavior = df.merge(repo_behavior_for_merge, on='repository', how='left')

repo_level_counts = (
    repo_counts['repo_reporting_behavior']
    .value_counts()
    .reindex(behavior_order)
    .rename_axis('Repository reporting behavior')
    .reset_index(name='Repositories')
)
repo_level_counts['Repo %'] = (
    repo_level_counts['Repositories'] / repo_counts['repository'].nunique() * 100
).round(2)

cve_level_counts = (
    df_with_repo_behavior['repo_reporting_behavior']
    .value_counts()
    .reindex(behavior_order)
    .rename_axis('Repository reporting behavior')
    .reset_index(name='CVEs')
)
cve_level_counts['CVE %'] = (
    cve_level_counts['CVEs'] / df_with_repo_behavior['CVE_ID'].nunique() * 100
).round(2)

severity_counts = (
    df_with_repo_behavior
    .pivot_table(
        index='repo_reporting_behavior',
        columns='Severity',
        values='CVE_ID',
        aggfunc='count',
        fill_value=0,
    )
    .reindex(behavior_order)
    .reset_index()
    .rename(columns={'repo_reporting_behavior': 'Repository reporting behavior'})
)

for col in ['Critical', 'High', 'Medium', 'Low', 'Unknown']:
    if col not in severity_counts.columns:
        severity_counts[col] = 0

publication_table = (
    repo_level_counts
    .merge(cve_level_counts, on='Repository reporting behavior', how='left')
    .merge(severity_counts[['Repository reporting behavior', 'Critical', 'High', 'Medium', 'Low', 'Unknown']],
           on='Repository reporting behavior', how='left')
)

publication_table


,Repository reporting behavior,Repositories,Repo %,CVEs,CVE %,Critical,High,Medium,Low,Unknown
0,Only transparent fixes,110,42.80,195,23.44,7,46,130,12,0
1,Only silent fixes,96,37.35,147,17.67,2,24,116,5,0
2,Mixed silent and transparent fixes,51,19.84,490,58.89,27,70,362,30,1


## Tables for Reporting


In [16]:
repo_behavior_summary


,Repository reporting behavior,Repository count,%,Severity distribution
0,Only transparent fixes,110,42.80,"Critical (7), High (46), Medium (130), Low (12)"
1,Only silent fixes,96,37.35,"Critical (2), High (24), Medium (116), Low (5)"
2,Mixed silent and transparent fixes,51,19.84,"Critical (27), High (70), Medium (362), Low (3..."


In [17]:
repo_counts


reporting_visibility,repository,Silent,Transparent,total_cves,repo_reporting_behavior
0,apache/tomcat,15,52,67,Mixed silent and transparent fixes
1,apache/struts,19,24,43,Mixed silent and transparent fixes
2,spring-projects/spring-framework,20,4,24,Mixed silent and transparent fixes
3,keycloak/keycloak,18,4,22,Mixed silent and transparent fixes
4,x-stream/xstream,15,7,22,Mixed silent and transparent fixes
...,...,...,...,...,...
252,wildfly/wildfly-naming-client,1,0,1,Only silent fixes
253,wso2/transport-http,1,0,1,Only silent fixes
254,xjodoin/torpedoquery,1,0,1,Only silent fixes
255,zeroturnaround/zt-zip,1,0,1,Only silent fixes


In [18]:
# Save nothing by default. Uncomment if needed.
# repo_behavior_summary.to_csv('repo_reporting_behavior_summary.csv', index=False)
# repo_counts.to_csv('repo_reporting_behavior_by_repo.csv', index=False)
